# Experiment 1 - Competing Drives

## Scenario

In this experiment we will simulate an agent which has access to food and shelter.

When not at food, it senses its **hunger** increasing with each action. When at food, it resets to zero.

Shelter is the same - when away, the agent senses its **risk** increasing with each action. When at shelter, it resets to zero.

The agent equally dislikes hunger and risk. It increases exponentially, so doesn't mind a little but really hates a lot (as it becomes 'life-endangering').

These competing drives cause it to run back and forth between the shelter and food as it becomes overwhelmed by its desire to eat or hide.


## Rewards

This simulation demonstrates purely **extrinsic** value driven behaviour - the agent sees a state within the policy horizon that provides closer alignment with its preferred state (C matrix).

The agent believes it has complete information about the state (it's A matrix is identity, observations map 1:1 with hidden states). 

This means it doesn't have an incentive to seek observations that would make it more certain about its true state (**epistemic** value) in order to help it reach the goal.


## Policy Length

In this simulation, it is the (mandatory, singular) Hunger action ('eat') taken *after* landing on the food that causes relief, not the one that lands you there.

Similarly it is the (mandatory, singular) Stress action ('hide') taken *after* landing on the shelter that causes relief, not the one that lands you there.

This means that the agent has to be able to imagine the state one timestep *after* landing on the relevant goal to anticipate a reward.

The minimum policy length must therefore be one more than the Manhatten distance (L1, shortest path without diagonal jumps).


> I tried setting a belief that acting to *land* on a goal would cause you to see relief as that would reduce the policy length by one. The agent expected to always observe 0 stress/hunger at the goal locations (A matrix). Similarly, I updated the drive state in the env *after* updating location (reward if you arrived). This seemed sensible but the problem is that the B matrix can only say e.g. 'Given my current hunger and location, if I eat, what will be my resulting hunger'. The answer would depend on whether the agent moved into the goal (reset) or away from the goal (increment). It seemed that the only way to be deterministic is to say 'you have to eat *at* the food location'.



In [2]:
%pip install inferactively-pymdp

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
from pymdp.agent import Agent
from pymdp import utils

## Hidden States

We will define a state space that incoporates
- Our **location** on a grid (index in list of coordinates)
- Our **hunger** level
- Our **stress** level

In [4]:
grid_dims = [4, 4]
n_grid_points = int(np.prod(grid_dims))
shelter_location = (0, 0)
food_location = (3, 0)
max_drive = 10

print(f"Grid dimensions: {grid_dims}")
print(f"Food location: {food_location}")
print(f"Shelter location: {shelter_location}")
print(f"Maximum drive level: {max_drive}")

Grid dimensions: [4, 4]
Food location: (3, 0)
Shelter location: (0, 0)
Maximum drive level: 10


In [5]:
# Create a look-up table `loc_list` that maps linear indices to (y, x) coordinates
grid = np.arange(n_grid_points).reshape(grid_dims)
it = np.nditer(grid, flags=["multi_index"])
loc_list = []
while not it.finished:
    loc_list.append(it.multi_index)
    it.iternext()

manhatten_distance = abs(food_location[0]-shelter_location[0]) + abs(food_location[1]-shelter_location[1])

# Without any information to gain (epistemic value) by moving, policies must be at least the manhattan distance from the goal to see any extrinsic value in moving.
print(f"Required policy length: {manhatten_distance + 1} steps")

Required policy length: 4 steps


In [6]:
hunger_levels = np.arange(max_drive + 1)  # 0 to max_drive, inclusive
stress_levels = np.arange(max_drive + 1) 
n_hunger = len(hunger_levels)
n_stress = len(stress_levels)

n_states = [
    n_grid_points,
    n_hunger,
    n_stress
]

print(f"Factor 0 (Location): {n_grid_points} states (one per grid cell)")
print(f"Factor 1 (Hunger): {n_hunger} states (0-{max_drive})")
print(f"Factor 2 (Stress): {n_stress} states (0-{max_drive})")

Factor 0 (Location): 16 states (one per grid cell)
Factor 1 (Hunger): 11 states (0-10)
Factor 2 (Stress): 11 states (0-10)


In [7]:
food_loc_idx = loc_list.index(food_location)
shelter_loc_idx = loc_list.index(shelter_location)
print(f"Shelter {shelter_location} = cell {shelter_loc_idx}\nFood {food_location} = cell {food_loc_idx}")

Shelter (0, 0) = cell 0
Food (3, 0) = cell 12


## A Matrix: Observation -> State

Observations are a simple 1:1 mapping to hidden states. 

In [8]:
# Create observation names for each factor to make it easier to interpret the agent's belief indices later on
location_obs_names = [f"({y},{x})" for y,x in loc_list]
hunger_obs_names = ['full'] + [f'hunger_{h}' for h in range(1, max_drive + 1)]
stress_obs_names = ['safe'] + [f'stress_{r}' for r in range(1, max_drive + 1)]
n_hunger_obs = len(hunger_obs_names)
n_stress_obs = len(stress_obs_names)

# Every observation maps 1:1 with a state so has the same dims as the state space
n_obs = [
    n_grid_points,
    n_hunger_obs,
    n_stress_obs
]

print(f"Observation Dimensionalities (Location, Hunger, Stress): {n_obs}")
print(f"Hunger Names: {hunger_obs_names}")
print(f"Stress Names: {stress_obs_names}")

Observation Dimensionalities (Location, Hunger, Stress): [16, 11, 11]
Hunger Names: ['full', 'hunger_1', 'hunger_2', 'hunger_3', 'hunger_4', 'hunger_5', 'hunger_6', 'hunger_7', 'hunger_8', 'hunger_9', 'hunger_10']
Stress Names: ['safe', 'stress_1', 'stress_2', 'stress_3', 'stress_4', 'stress_5', 'stress_6', 'stress_7', 'stress_8', 'stress_9', 'stress_10']


In this experiment to keep it simple every observation modality is indexed against every state factor, whether or not it is useful.

> In later experiments we will filter so that observations are only mapped to state factors that they depend on.

In [9]:
# For each modality, its observation count followed by the counts of every hidden states.
A = utils.obj_array_zeros([[n_obs[m]] + n_states for m in range(len(n_obs))])

print(f"A matrix shapes")
print(f"  A[0] Location: {A[0].shape}")
print(f"  A[1] Hunger sensor: {A[1].shape}")
print(f"  A[2] Stress sensor: {A[2].shape}")

A matrix shapes
  A[0] Location: (16, 16, 11, 11)
  A[1] Hunger sensor: (11, 16, 11, 11)
  A[2] Stress sensor: (11, 16, 11, 11)


Each observation maps 1:1 with the corresponding state factor, i.e. we can sense the hidden states perfectly.

This means we need identity A matrices for each modality.

In [10]:
for loc_idx in range(n_grid_points):
    for h in range(n_hunger):
        for r in range(n_stress):
            A[0][loc_idx, loc_idx, h, r] = 1.0
            A[1][h, loc_idx, h, r] = 1.0
            A[2][r, loc_idx, h, r] = 1.0

## Normalisation

The probabilities of possible observations given a state must sum to 1 - i.e. you can't be 60% confident in observing X *and* 50% confident in observing Y. You have to split your probability assignment between the available. options.

In [11]:
for m in range(len(A)):
    sum_over_obs = A[m].sum(axis=0) # Sum over the observation axis for modality m
    is_norm_m = np.allclose(sum_over_obs, 1.0)
    if not is_norm_m:
        print(f"Warning: A[{m}] is NOT normalized!")

## B Matrix: State -> Action -> State

We can start by defining
- Action counts per state factor
- Which factors have user-controllable actions
- Dependencies between factors

In [12]:
n_controls = [5, 1, 1]  # 5 movement actions, 1 null action for hunger, 1 null action for stress

# B_factor_list specifies which state factors each B matrix depends on:
# - B[0] (Location): depends only on Location (factor 0) - standard
# - B[1] (Hunger): depends on Location (factor 0) AND Hunger (factor 1) - location-dependent!
# - B[2] (Stress): depends on Location (factor 0) AND Stress (factor 2) - location-dependent!
B_factor_list = [[0], [0, 1], [0, 2]]

# Only Location (factor 0) is controllable by agent
control_fac_idx = [0]

Now we can initialize B matrices with appropriate shapes.
B for a given factor must be of the shape (next_state, *parent_factor_dims, num_actions)

In [13]:
B = utils.obj_array(len(n_states))
B[0] = np.zeros((n_grid_points, n_grid_points, n_controls[0]))  # (16, 16, 5)
B[1] = np.zeros((n_hunger, n_grid_points, n_hunger, n_controls[1]))  # (6, 16, 6, 1)
B[2] = np.zeros((n_stress, n_grid_points, n_stress, n_controls[2]))  # (6, 16, 6, 1)

print(f"B dependencies: {B_factor_list}")
print(f"Controllable factors: {control_fac_idx}")
print(f"B[0] Location: {B[0].shape} - depends on [Location (0)] with 5 movement actions")
print(f"B[1] Hunger: {B[1].shape} - depends on [Location (0), Hunger (1)] with 1 null action")
print(f"B[2] Stress: {B[2].shape} - depends on [Location (0), Stress (2)] with 1 null action")

B dependencies: [[0], [0, 1], [0, 2]]
Controllable factors: [0]
B[0] Location: (16, 16, 5) - depends on [Location (0)] with 5 movement actions
B[1] Hunger: (11, 16, 11, 1) - depends on [Location (0), Hunger (1)] with 1 null action
B[2] Stress: (11, 16, 11, 1) - depends on [Location (0), Stress (2)] with 1 null action


### Location Transitions

To initialise the location element of the B matrix we will
- Iterate through each of the 5 possible actions (UP, DOWN, LEFT, RIGHT, STAY).
- For each action and location, calculate the next location accounting for grid boundaries.
- Set `B[0][next_loc, curr_loc, action_id] = 1.0` for deterministic transitions.

In [14]:
actions = ["UP", "DOWN", "LEFT", "RIGHT", "STAY"]

for action_id, action_label in enumerate(actions):
    for curr_loc_idx in range(n_grid_points):
        curr_y, curr_x = loc_list[curr_loc_idx]
        
        # Compute next location based on action
        if action_label == "UP":
            next_y = curr_y - 1 if curr_y > 0 else curr_y
            next_x = curr_x
        elif action_label == "DOWN":
            next_y = curr_y + 1 if curr_y < (grid_dims[0]-1) else curr_y
            next_x = curr_x
        elif action_label == "LEFT":
            next_y = curr_y
            next_x = curr_x - 1 if curr_x > 0 else curr_x
        elif action_label == "RIGHT":
            next_y = curr_y
            next_x = curr_x + 1 if curr_x < (grid_dims[1]-1) else curr_x
        elif action_label == "STAY":
            next_y, next_x = curr_y, curr_x
        
        next_loc = (next_y, next_x)
        next_loc_idx = loc_list.index(next_loc)
        
        # Set transition probability
        B[0][next_loc_idx, curr_loc_idx, action_id] = 1.0

As before, your beliefs about your resulting state given a starting state and an action must sum to 1.

In [15]:
for action_id in range(5):
    col_sums = B[0][:, :, action_id].sum(axis=0)
    if not np.allclose(col_sums, 1.0):
        print(f"  Warning: Location probabilities in B[0][:,:,{action_id}] not normalized!")

### Hunger and Stress Transitions

There is only one 'null' action for the Hunger and Stress modalities which isn't user controllable.

In this case we can imagine it as a compulsory attempt to eat and hide on every timestep, as these reduce states which we dislike.

The result of the eat and hide actions depend on your location.

If you are at the relevant goal, you succeed and transition to 0 for that modality.

If not, you transition to one point higher in that modality up until the maximum.

> Parameter note: If the agent can see the goal but doesn't have max_stress 'headroom' then it will die whether it moves or not, so has no motivation to move.

Their B matrices will be of the shape: `(max_drive, max_drive, n_grid_points, 1)`.

In [16]:
for loc_idx in range(n_grid_points):
    for curr_hunger in range(n_hunger):
        if loc_idx == food_loc_idx:
            # AT FOOD: hunger resets to 0 regardless of current hunger
            next_hunger = 0
        else:
            # ELSEWHERE: hunger increases by 1, capped at max
            next_hunger = min(curr_hunger + 1, max_drive)
        
        # Single null action (action_id=0)
        B[1][next_hunger, loc_idx, curr_hunger, 0] = 1.0

print(f"  At food (loc {food_loc_idx}): hunger -> 0")
print(f"  Elsewhere: hunger -> min(hunger+1, {max_drive})")


for loc_idx in range(n_grid_points):
    for curr_stress in range(n_stress):
        if loc_idx == shelter_loc_idx:
            # AT SHELTER: stress resets to 0 regardless of current stress
            next_stress = 0
        else:
            # ELSEWHERE: stress increases by 1, capped at max
            next_stress = min(curr_stress + 1, max_drive)
        
        # Single null action (action_id=0)
        B[2][next_stress, loc_idx, curr_stress, 0] = 1.0

print(f"  At shelter (loc {shelter_loc_idx}): stress -> 0")
print(f"  Elsewhere: stress -> min(stress+1, {max_drive})")      

  At food (loc 12): hunger -> 0
  Elsewhere: hunger -> min(hunger+1, 10)
  At shelter (loc 0): stress -> 0
  Elsewhere: stress -> min(stress+1, 10)


In [18]:
for f, factor_name in enumerate(['Location', 'Hunger', 'Stress']):
    # Check that columns sum to 1 for all control/parent configurations
    B_f = B[f]
    # Sum over first axis (next state)
    total_axes = tuple(range(1, len(B_f.shape)))  # All axes except the first
    for idx in np.ndindex(B_f.shape[1:]):
        col_sum = B_f[(slice(None),) + idx].sum()
        if not np.isclose(col_sum, 1.0):
            print(f"  Warning: B[{f}] column at {idx} sums to {col_sum:.3f}")

## C Matrix: Preferred State

The C matrix isn't in the 0-1 probabilities range of the M matrix observations but arbitrary magnitudes which are used as log probabilities.

This is more intuitive as it linearly scales like a 'reward' or 'punishment' score.

- The agent has no inherent preference for location, so is all zeroes.
- Hunger and Stress scale exponentially with a huge jump for max_stress ('die').



In [20]:
C = utils.obj_array_zeros(n_obs)

penalty_max = -50.0
exp_steepness = 4

# Hunger Preferences
n_hunger = len(C[1])
C[1][-1] = penalty_max # max hunger - CATASTROPHIC!
if n_hunger > 2:
    for i in range(0, n_hunger): # Max already set, we are filling from 0 to max-1
        denominator = (n_hunger) - 1
        progress = i / denominator if denominator > 0 else 1.0 # Progress from 0 to 1 across the hunger levels (excluding max)
        
        # Exponential scaling: (e^(kx) - 1) / (e^k - 1)
        # Result: Low penalty for low hunger, massive penalty for high hunger
        scale = (np.exp(exp_steepness * progress) - 1) / (np.exp(exp_steepness) - 1)
        C[1][i] = penalty_max * scale

# Stress Preferences - symmetrical with hunger
n_stress = len(C[2])
C[2][-1] = penalty_max
if n_stress > 2:
    for i in range(0, n_stress - 1):
        denominator = (n_stress) - 1
        progress = i / denominator if denominator > 0 else 1.0
        scale = (np.exp(exp_steepness * progress) - 1) / (np.exp(exp_steepness) - 1)
        C[2][i] = penalty_max * scale

print("C (prior preferences) vector filled.")
print(f"Hunger preferences: {np.round(C[1], 2)}")
print(f"Stress preferences: {np.round(C[2], 2)}")
print("\nPreferences scaled automatically to state space dimensions.")

C (prior preferences) vector filled.
Hunger preferences: [ -0.    -0.46  -1.14  -2.16  -3.69  -5.96  -9.35 -14.41 -21.95 -33.21
 -50.  ]
Stress preferences: [ -0.    -0.46  -1.14  -2.16  -3.69  -5.96  -9.35 -14.41 -21.95 -33.21
 -50.  ]

Preferences scaled automatically to state space dimensions.


## D Matrix : Starting State Belief

We want the agent to start with correct beliefs, so whatever we encode here should match the starting environment.

We will start at the centre of the grid with zero hunger and stress.

In [ ]:
# One prior per factor
D = utils.obj_array_uniform(n_states)

# Set agent's initial beliefs:
# - Location: center of grid (2,2)
# - Hunger: 0 (satiated)
# - Stress: 0 (safe)
start_loc = (2, 2)
start_loc_idx = loc_list.index(start_loc)
start_hunger = 0
start_stress = 0

# One-hot vectors for each factor - we precisely belive we will start with the values defined above.
D[0] = utils.onehot(start_loc_idx, n_grid_points)
D[1] = utils.onehot(start_hunger, n_hunger)
D[2] = utils.onehot(start_stress, n_stress)

print("D[0] Location prior shape: ", D[0].shape)
print("D[1] Hunger prior shape: ", D[1].shape)
print("D[2] Stress prior shape: ", D[2].shape)
print(f"D[0] Location: starts at {start_loc} (idx {start_loc_idx})")
print(f"D[1] Hunger: starts at {start_hunger}")
print(f"D[2] Stress: starts at {start_stress}")

D[0] Location prior shape:  (16,)
D[1] Hunger prior shape:  (11,)
D[2] Stress prior shape:  (11,)
D[0] Location: starts at (2, 2) (idx 10)
D[1] Hunger: starts at 0
D[2] Stress: starts at 0


## Generative Process: The Environment